# Cuaderno 5 - Filtrar filas con `filter()`

## Antes de comenzar

En el cuaderno anterior utilizamos `select()` para construir nuevos DataFrames a partir de columnas y expresiones. Ahora vamos a incorporar otra operación fundamental: **conservar solamente las filas que cumplen una condición**.

Trabajaremos con `filter()` y con expresiones booleanas como:

```python
pl.col("precio") > 5000
```

La intención es observar con claridad qué diferencia existe entre una expresión booleana y el uso de esa expresión como criterio de filtrado.

Seguiremos utilizando el pequeño conjunto de ventas de la tienda escolar, de modo que cada resultado pueda compararse fácilmente con la tabla original.


## 1. Preparar el entorno y los datos

Instalamos Polars en el entorno de Colab e importamos la biblioteca con la abreviatura habitual `pl`.


In [45]:
!pip install -q polars

In [46]:
import polars as pl

Creamos nuevamente el DataFrame de ventas que utilizamos en los capítulos anteriores.


In [47]:
datos = {
    "producto": ["Cuaderno", "Lapicera", "Mochila", "Calculadora", "Regla", "Carpeta"],
    "categoria": ["Librería", "Librería", "Accesorios", "Tecnología", "Librería", "Librería"],
    "precio": [3500, 1200, 28500, 18500, 900, 4200],
    "cantidad": [4, 10, 2, 3, 6, 5],
}

df = pl.DataFrame(datos)
df

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


El DataFrame ejecutado tiene `shape: (6, 4)`: contiene seis filas y cuatro columnas, llamadas `producto`, `categoria`, `precio` y `cantidad`. A partir de esta tabla construiremos distintos filtros sin modificar el objeto original.


## 2. Construir una condición booleana

Comenzamos con una condición sencilla: queremos saber qué filas tienen un precio mayor que `5000`.


In [48]:
pl.col("precio") > 5000

<Expr ['[(col("precio")) > (dyn int: 5…'] at 0x7BF9BCECE650>

La salida muestra un objeto `Expr`, no una serie de valores `True` o `False`. La comparación:

```python
pl.col("precio") > 5000
```

todavía describe una condición sobre una columna llamada `precio`; no fue evaluada sobre `df`.

Para observar el resultado fila por fila necesitamos utilizar esa expresión dentro de un contexto.


Podemos evaluar la condición dentro de `select()` para observar su resultado fila por fila.


In [49]:
df.select(
    (pl.col("precio") > 5000).alias("precio_mayor_5000")
)

precio_mayor_5000
bool
false
false
true
true
false
false


Dentro de `select()`, la expresión se materializa como una columna booleana de seis filas. Los valores son `false`, `false`, `true`, `true`, `false` y `false`.

Las dos posiciones que producen `true` corresponden a `Mochila` y `Calculadora`, cuyos precios superan los `5000`. Esta salida nos permite observar directamente el resultado de la condición antes de utilizarla como filtro.


## 3. Utilizar la condición dentro de `filter()`

La misma expresión puede utilizarse directamente dentro de `filter()`.


In [50]:
df.filter(
    pl.col("precio") > 5000
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3


La salida tiene `shape: (2, 4)` y conserva únicamente `Mochila` y `Calculadora`. Son las dos filas para las cuales la expresión `pl.col("precio") > 5000` produjo `true`.

Las demás filas no forman parte de este resultado, aunque continúan presentes en `df`.


## 4. Comparaciones numéricas

Podemos construir filtros utilizando los operadores de comparación habituales.

Comenzamos buscando productos con un precio menor que `2000`.


In [51]:
df.filter(
    pl.col("precio") < 2000
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Lapicera""","""Librería""",1200,10
"""Regla""","""Librería""",900,6


La salida tiene dos filas: `Lapicera`, con precio `1200`, y `Regla`, con precio `900`. Son los únicos registros cuyo valor de `precio` es menor que `2000`.


Ahora incluimos también el límite mediante `<=`.


In [52]:
df.filter(
    pl.col("precio") <= 3500
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Regla""","""Librería""",900,6


La salida contiene tres filas: `Cuaderno`, `Lapicera` y `Regla`. `Cuaderno` queda incluido porque su precio es exactamente `3500`, lo que permite observar la diferencia entre `<` y `<=`.


Podemos aplicar el mismo criterio sobre otra columna. Busquemos ahora los productos cuya cantidad sea mayor o igual que `5`.


In [53]:
df.filter(
    pl.col("cantidad") >= 5
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Lapicera""","""Librería""",1200,10
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


La salida vuelve a tener tres filas: `Lapicera`, `Regla` y `Carpeta`, con cantidades `10`, `6` y `5`. En este caso la condición se aplicó sobre `cantidad` en lugar de `precio`.


## 5. Filtrar por igualdad

También podemos trabajar con columnas de texto.

Busquemos todos los productos pertenecientes a la categoría `Librería`.


In [54]:
df.filter(
    pl.col("categoria") == "Librería"
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


La salida tiene `shape: (4, 4)` y conserva `Cuaderno`, `Lapicera`, `Regla` y `Carpeta`. Los cuatro registros poseen exactamente el valor `"Librería"` en la columna `categoria`.


Podemos realizar la operación inversa utilizando `!=`.


In [55]:
df.filter(
    pl.col("categoria") != "Librería"
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3


La condición inversa deja un resultado de dos filas: `Mochila`, cuya categoría es `Accesorios`, y `Calculadora`, cuya categoría es `Tecnología`.


## 6. Guardar una condición en una variable

Como las condiciones son expresiones, podemos almacenarlas y reutilizarlas.


In [56]:
es_caro = pl.col("precio") > 5000
es_caro

<Expr ['[(col("precio")) > (dyn int: 5…'] at 0x7BF9BCECEC50>

La salida de `es_caro` vuelve a mostrar un objeto `Expr`. Guardar la condición en una variable no la evalúa: `es_caro` sigue representando la comparación `precio > 5000`.


In [57]:
df.filter(es_caro)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3


Al utilizar `es_caro` dentro de `filter()`, la expresión se evalúa sobre `df` y obtenemos nuevamente un DataFrame de dos filas: `Mochila` y `Calculadora`.

La condición pudo construirse una vez y utilizarse después sin necesidad de escribir nuevamente la comparación.


## 7. La misma expresión en dos contextos

Podemos utilizar exactamente la misma condición dentro de `select()` y dentro de `filter()`.


In [58]:
condicion = pl.col("precio") > 5000

df.select(
    condicion.alias("es_mayor_5000")
)

es_mayor_5000
bool
false
false
true
true
false
false


Dentro de `select()`, la condición se transforma en una columna booleana de seis valores: `false`, `false`, `true`, `true`, `false` y `false`. En este contexto estamos observando el resultado de la expresión, no eliminando filas.


In [59]:
df.filter(condicion)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3


Al utilizar exactamente la misma expresión dentro de `filter()`, el resultado cambia de función: ahora obtenemos un DataFrame de `shape: (2, 4)` con `Mochila` y `Calculadora`.

La expresión es la misma; lo que cambia es el contexto en el que Polars la evalúa.


## 8. Filtrar utilizando una lista de valores

Cuando queremos conservar varias categorías posibles, `is_in()` permite expresar la pertenencia a una colección de valores.


In [60]:
df.filter(
    pl.col("categoria").is_in(
        ["Librería", "Tecnología"]
    )
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


La salida tiene cinco filas. Conserva los cuatro productos de `Librería` y también `Calculadora`, que pertenece a `Tecnología`. `Mochila` queda fuera porque `Accesorios` no aparece en la lista proporcionada a `is_in()`.


## 9. Filtrar dentro de un intervalo

Podemos utilizar `is_between()` para comprobar si los valores de una columna se encuentran dentro de un rango.


In [61]:
df.filter(
    pl.col("precio").is_between(1000, 5000)
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Carpeta""","""Librería""",4200,5


La salida tiene tres filas: `Cuaderno`, `Lapicera` y `Carpeta`, con precios `3500`, `1200` y `4200`.

`Regla`, con `900`, queda por debajo del límite inferior; `Mochila` y `Calculadora`, con `28500` y `18500`, superan el límite superior. En este ejemplo los extremos del intervalo se consideran incluidos.


## 10. Un filtro que no encuentra filas

También podemos construir una condición que ningún registro cumpla.


In [62]:
df.filter(
    pl.col("precio") > 100000
)

producto,categoria,precio,cantidad
str,str,i64,i64


La salida tiene `shape: (0, 4)`: no contiene ninguna fila, pero conserva las cuatro columnas originales y sus tipos.

No ocurrió ningún error. El filtro fue evaluado correctamente y simplemente no encontró registros cuyo precio superara `100000`.


## 11. Un filtro que conserva todas las filas

Veamos ahora el caso opuesto.


In [63]:
df.filter(
    pl.col("precio") > 0
)

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


La salida vuelve a tener `shape: (6, 4)`. Como todos los precios son mayores que cero, las seis filas cumplen la condición y permanecen en el resultado.

`filter()` no implica necesariamente reducir la tabla: conserva todas las filas cuya condición produzca `true`, sean ninguna, algunas o todas.


## 12. Encadenar `filter()` y `select()`

Podemos combinar los dos contextos que ya conocemos.

Queremos conservar únicamente los productos con precio mayor que `5000` y, después, mostrar solamente `producto` y `precio`.


In [64]:
df.filter(
    pl.col("precio") > 5000
).select(
    "producto",
    "precio",
)

producto,precio
str,i64
"""Mochila""",28500
"""Calculadora""",18500


La salida tiene `shape: (2, 2)` y contiene únicamente `Mochila` con precio `28500` y `Calculadora` con precio `18500`.

La cadena puede leerse de izquierda a derecha: primero `filter()` conserva las dos filas cuyo precio supera `5000`; después `select()` reduce el resultado a las columnas `producto` y `precio`.


## 13. El orden de las operaciones importa

También podemos seleccionar primero las columnas necesarias y filtrar después.


In [65]:
df.select(
    "producto",
    "precio",
).filter(
    pl.col("precio") > 5000
)

producto,precio
str,i64
"""Mochila""",28500
"""Calculadora""",18500


La salida vuelve a ser exactamente la misma: dos filas y dos columnas, con `Mochila` y `Calculadora`.

Esto funciona porque `precio` continúa disponible después del primer `select()`. Si hubiéramos seleccionado solamente `producto`, la operación siguiente ya no podría construir una condición basada en `precio`. Cada paso trabaja con la estructura que recibe del anterior.


## 14. El DataFrame original continúa intacto

Volvemos a observar `df`.


In [66]:
df

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


La salida final vuelve a mostrar `shape: (6, 4)` y los seis registros originales. Ninguno de los filtros anteriores modificó `df`; cada llamada a `filter()` produjo un nuevo DataFrame.


## 15. Qué conviene recordar

En este cuaderno incorporamos `filter()` como el contexto que permite conservar filas a partir de una expresión booleana.

Trabajamos con comparaciones como:

```text
>
<
>=
<=
==
!=
```

y también con:

```text
is_in()
is_between()
```

Observamos además que una misma expresión puede utilizarse en distintos contextos: dentro de `select()` produce una columna de valores booleanos y dentro de `filter()` decide qué filas permanecen.

Todavía utilizamos una sola condición por vez. Esto fue intencional: antes de combinar reglas conviene entender bien cómo se construye y evalúa una condición simple.


## Próximo paso

Muchas preguntas reales necesitan combinar más de una condición.

Podemos querer, por ejemplo, productos de `Librería` **y** con precio mayor que `2000`, productos de `Tecnología` **o** `Accesorios`, o todos los registros que **no** cumplen cierta regla.

En el próximo capítulo trabajaremos con `&`, `|` y `~` para combinar expresiones booleanas y veremos por qué los paréntesis son una parte importante de la lógica de esos filtros.


---

### Autoría y atribución

Este cuaderno forma parte del material educativo desarrollado por **VintaBytes**.

📧 **Contacto:** [vintabytes@gmail.com](mailto:vintabytes@gmail.com)

🌐 **Repositorio oficial:** [github.com/VintaBytes/Ciencia-de-datos](https://github.com/VintaBytes/Ciencia-de-datos)

Se permite compartir y reutilizar este material respetando la autoría y manteniendo visible esta atribución.

---
